<a href="https://colab.research.google.com/github/burhanuddin-down/sensitive-java-script-files-founder/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [2]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [3]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
TENNIS_PLAYERS = {

    "novak_djokovic": (
        "Name: Novak Djokovic"
        "Country: Serbia"
        "Playing Style: Right-handed, two-handed backhand"
        "Known For: Exceptional return of serve, defensive ability, flexibility, and mental strength"
        "Grand Slam Titles: 24"
        "Major Achievements: Multiple Australian Open, French Open, Wimbledon, and US Open titles"
        "Career Highlights: One of the most successful men's singles players in tennis history"
        "Surface Strength: Hard court, grass, and clay"
    ),

    "roger_federer": (
        "Name: Roger Federer"
        "Country: Switzerland"
        "Playing Style: Right-handed, one-handed backhand"
        "Known For: Elegant technique, powerful serve, precise footwork, and all-court play"
        "Grand Slam Titles: 20"
        "Major Achievements: Multiple Wimbledon, Australian Open, US Open, and French Open titles"
        "Career Highlights: Spent 310 weeks as world No. 1, including 237 consecutive weeks"
        "Surface Strength: Grass and hard court"
    ),

    "rafael_nadal": (
        "Name: Rafael Nadal"
        "Country: Spain"
        "Playing Style: Left-handed, two-handed backhand"
        "Known For: Powerful topspin, exceptional defense, stamina, and competitive intensity"
        "Grand Slam Titles: 22"
        "Major Achievements: 14 French Open titles and multiple Australian Open, Wimbledon, and US Open titles"
        "Career Highlights: One of the greatest clay-court players in tennis history"
        "Surface Strength: Clay"
    ),

    "serena_williams": (
        "Name: Serena Williams"
        "Country: United States"
        "Playing Style: Right-handed, two-handed backhand"
        "Known For: Powerful serve, aggressive baseline game, athleticism, and mental strength"
        "Grand Slam Singles Titles: 23"
        "Major Achievements: Multiple Australian Open, French Open, Wimbledon, and US Open titles"
        "Career Highlights: One of the most successful women's singles players of the Open Era"
        "Surface Strength: Hard court and grass"
    ),

    "venus_williams": (
        "Name: Venus Williams"
        "Country: United States"
        "Playing Style: Right-handed, two-handed backhand"
        "Known For: Powerful serve, attacking baseline play, athleticism, and aggressive tennis"
        "Grand Slam Singles Titles: 7"
        "Major Achievements: Five Wimbledon singles titles and two US Open singles titles"
        "Career Highlights: Former world No. 1 and multiple Olympic gold medalist"
        "Surface Strength: Grass and hard court"
    ),

    "martina_navratilova": (
        "Name: Martina Navratilova"
        "Country: Czech Republic / United States"
        "Playing Style: Left-handed, one-handed backhand"
        "Known For: Serve-and-volley tennis, exceptional fitness, and aggressive net play"
        "Grand Slam Singles Titles: 18"
        "Major Achievements: Nine Wimbledon singles titles"
        "Career Highlights: One of the greatest women's players and doubles players in tennis history"
        "Surface Strength: Grass"
    ),

    "steffi_graf": (
        "Name: Steffi Graf"
        "Country: Germany"
        "Playing Style: Right-handed, one-handed backhand"
        "Known For: Powerful forehand, speed, footwork, and aggressive baseline play"
        "Grand Slam Singles Titles: 22"
        "Major Achievements: Won all four Grand Slam tournaments in 1988"
        "Career Highlights: Achieved the Golden Slam in 1988"
        "Surface Strength: Hard court, clay, and grass"
    ),

    "carlos_alcaraz": (
        "Name: Carlos Alcaraz"
        "Country: Spain"
        "Playing Style: Right-handed, two-handed backhand"
        "Known For: Explosive movement, powerful forehand, drop shots, and aggressive all-court play"
        "Grand Slam Titles: Multiple Grand Slam singles titles"
        "Major Achievements: Won Grand Slam titles on multiple surfaces"
        "Career Highlights: Became one of the youngest players to reach world No. 1"
        "Surface Strength: Hard court, clay, and grass"
    ),

    "jannik_sinner": (
        "Name: Jannik Sinner"
        "Country: Italy"
        "Playing Style: Right-handed, two-handed backhand"
        "Known For: Powerful groundstrokes, aggressive baseline play, movement, and consistency"
        "Grand Slam Titles: Multiple Grand Slam singles titles"
        "Major Achievements: Established himself among the leading players of the modern era"
        "Career Highlights: Reached world No. 1 and won major titles"
        "Surface Strength: Hard court"
    ),

    "andy_murray": (
        "Name: Andy Murray"
        "Country: United Kingdom"
        "Playing Style: Right-handed, two-handed backhand"
        "Known For: Defensive skills, counterpunching, tactical intelligence, and exceptional returning"
        "Grand Slam Singles Titles: 3"
        "Major Achievements: Two Wimbledon titles and one US Open title"
        "Career Highlights: Former world No. 1 and multiple Olympic gold medalist"
        "Surface Strength: Grass and hard court"
    ),

    "pete_sampras": (
        "Name: Pete Sampras"
        "Country: United States"
        "Playing Style: Right-handed, one-handed backhand"
        "Known For: Powerful serve, serve-and-volley play, and attacking tennis"
        "Grand Slam Singles Titles: 14"
        "Major Achievements: Seven Wimbledon titles and five US Open titles"
        "Career Highlights: Dominated men's tennis during much of the 1990s"
        "Surface Strength: Grass and hard court"
    ),

    "billie_jean_king": (
        "Name: Billie Jean King"
        "Country: United States"
        "Playing Style: Right-handed"
        "Known For: Aggressive net play, tactical intelligence, and strong doubles ability"
        "Grand Slam Singles Titles: 12"
        "Major Achievements: Six Wimbledon singles titles and four US Open singles titles"
        "Career Highlights: One of the most influential players in women's tennis history"
        "Surface Strength: Grass and hard court"
    )

}

In [10]:
def search_documents(query: str) -> str:
    """Search the tennis player collection for information relevant to the query."""

    query_lower = query.lower()
    results = []

    for player_name, content in TENNIS_PLAYERS.items():

        words = set(query_lower.split())

        match_count = sum(
            1
            for word in words
            if word in content.lower()
        )

        if match_count > 0:
            results.append(
                (match_count, player_name, content)
            )

    # Sort by number of matching words
    results.sort(reverse=True)

    if not results:
        return "No relevant tennis players found for that query."

    # Return the top two matches
    output = []

    for _, name, content in results[:2]:
        output.append(
            f"[Source: {name}]\n{content}"
        )

    return "\n\n".join(output)


# Test the search function

print(search_documents("Grand Slam"))


[Source: venus_williams]
Name: Venus WilliamsCountry: United StatesPlaying Style: Right-handed, two-handed backhandKnown For: Powerful serve, attacking baseline play, athleticism, and aggressive tennisGrand Slam Singles Titles: 7Major Achievements: Five Wimbledon singles titles and two US Open singles titlesCareer Highlights: Former world No. 1 and multiple Olympic gold medalistSurface Strength: Grass and hard court

[Source: steffi_graf]
Name: Steffi GrafCountry: GermanyPlaying Style: Right-handed, one-handed backhandKnown For: Powerful forehand, speed, footwork, and aggressive baseline playGrand Slam Singles Titles: 22Major Achievements: Won all four Grand Slam tournaments in 1988Career Highlights: Achieved the Golden Slam in 1988Surface Strength: Hard court, clay, and grass


In [12]:
import google.generativeai as genai
import time
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-pro",
    tools=[search_documents],
    system_instruction=(
        '''
        You are a tennis expert. When you need specific information about tennis players,
        their playing style, grand slam titles, or career highlights, use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-pro


In [13]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_recipes(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Add a question here.",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)